In [ ]:
%run ./imports.py

### Load attacks, profiling and coverage data

In [ ]:
prefix_optimal_attacks_path = "data/prefix_optimal_attacks.pkl"
prefix_rtt_profiling_fixed_windows_path = "data/prefix_rtt_profiling_fixed_windows_path.pkl"
prefix_rtt_profiling_variable_windows_path = "data/prefix_rtt_profiling_variable_windows_path.pkl"
covered_prefixes_variable_windows_path = "data/covered_prefixes_variable_windows.pkl"
covered_prefixes_fixed_windows_path = "data/covered_prefixes_fixed_windows.pkl"

In [ ]:
df_prefix_optimal_attacks = pd.read_pickle(prefix_optimal_attacks_path)
df_prefix_optimal_attacks.head(n=1)

In [ ]:
absolute_thresholds = {
    prefix: group.set_index("Attacker")["PostAttack_ms"].to_dict()
    for prefix, group in df_prefix_optimal_attacks.groupby("Prefix")
}

In [ ]:
df_prefixes_profiling_variablewindows = pd.read_pickle(prefix_rtt_profiling_variable_windows_path)
df_prefixes_profiling_variablewindows.head(n=1)

In [ ]:
df_prefixes_profiling_fixedwindows = pd.read_pickle(prefix_rtt_profiling_fixed_windows_path)
df_prefixes_profiling_fixedwindows.head(n=1)

In [ ]:
geodesic_thresholds_varwin = df_prefixes_profiling_variablewindows.set_index('Destination_Prefix')["minRTT_Lower_Bound_ms"].to_dict()
geodesic_thresholds_fxdwin = df_prefixes_profiling_variablewindows.set_index('Destination_Prefix')["minRTT_Lower_Bound_ms"].to_dict()

In [ ]:
detection_phase_samples_varwin = dict(zip(
    df_prefixes_profiling_variablewindows["Destination_Prefix"],
    df_prefixes_profiling_variablewindows[["ACK_Timestamp_Detection", "RTT_ms_Detection"]].values.tolist()
))
detection_phase_samples_fxdwin = dict(zip(
    df_prefixes_profiling_fixedwindows["Destination_Prefix"],
    df_prefixes_profiling_fixedwindows[["ACK_Timestamp_Detection", "RTT_ms_Detection"]].values.tolist()
))

In [ ]:
with open(covered_prefixes_variable_windows_path, "rb") as fp:
    covered_prefixes_varwin = pickle.load(fp)
with open(covered_prefixes_fixed_windows_path, "rb") as fp:
    covered_prefixes_fxdwin = pickle.load(fp)

In [ ]:
MIN_WINDOW_SAMPLE_COUNT = 5

## Variable windows-based detection

In [ ]:
def _ceil(v):
    iv = int(v)
    # Ceil for non-integer floats; ints unchanged
    return iv if v == iv else iv + 1

def _floor(v):
    return int(v)

def detect_attacks_using_variable_windows(
    timestamps, rtts, A, S, G, N
):
    """
    Streaming/on-line detection of RTT surges per-prefix,
    window-based ambiguous-state tracking, ignoring samples < G,
    windows closing only on 0.25s grid boundaries,
    and tracking attack onset (first rtt >= A).

    Additional tracking fields:
      - samples_to_detect: RTT samples from attack_start to detection
      - samples_in_recovery: RTT samples from detection to recovery
      - sample_count: RTT samples in each ambiguous period

    Recovery & ambiguous durations = #slots * 0.25s.

    A window qualifies for detection if:
      (1) closed by ≥N samples within ≤60s, or
      (2) closed by timeout (<N samples, ≥60s) with minRTT < A-S.

    Ambiguous windows (minRTT ∈ [A-S, A)) are tracked but skip detection.

    Returns:
      { 'attacks': [
            { attack_start, suspected, detected, recovered,
              duration, detection_delay,
              samples_to_detect, samples_in_recovery }
          ],
        'ambiguous_periods': [
            { start, end, duration, sample_count }
        ] }
    """
    assert len(timestamps) == len(rtts), "timestamps/rtts must align"

    state = 'normal'
    prev_window_min = None
    suspect_time = None
    detect_time = None
    attack_start_ts = None

    samples_to_detect = 0
    samples_in_recovery = 0

    attacks = []
    ambiguous_periods = []

    recovery_slots = set()
    ambiguous_slots = set()

    amb_start = None
    amb_end = None

    window_start = None
    window_rtts = []
    window_tss = []
    last_window_ts = None

    for ts, rtt in zip(timestamps, rtts):
        if rtt < G:
            continue

        # Track attack onset samples
        if rtt >= A and attack_start_ts is None and state != 'detected':
            attack_start_ts = ts
            samples_to_detect = 1
        elif attack_start_ts is not None and detect_time is None:
            samples_to_detect += 1

        # Count recovery samples
        if state == 'detected':
            samples_in_recovery += 1

        # Attempt window-close on new sample
        if window_start is not None and last_window_ts is not None:
            times = []
            if len(window_rtts) >= N:
                times.append(_ceil(last_window_ts * 4) / 4.0)
            times.append(window_start + 60.0)
            close_time = min(times)

            if ts >= close_time:
                curr_min = min(window_rtts)
                num_samples = len(window_rtts)
                elapsed = close_time - window_start
                qualifies = (
                    (num_samples >= N and elapsed <= 60.0)
                    or
                    (num_samples < N and elapsed >= 60.0 and curr_min < (A - S))
                )

                if qualifies:
                    # ambiguous-slot tracking
                    slots = {_floor(t*4) for t in window_tss}

                    # Ambiguous window
                    if (A - S) <= curr_min < A:
                        if amb_start is None:
                            amb_start = window_start
                        amb_end = close_time
                        ambiguous_slots.update(slots)
                        # sample_count by num_samples
                        # (will record at exit)
                    else:
                        # exit ambiguous run if any
                        if amb_start is not None:
                            ambiguous_periods.append({
                                'start': amb_start,
                                'end': amb_end,
                                'duration': len(ambiguous_slots) * 0.25,
                                'sample_count': sum(
                                    1 for t in window_tss if (A - S) <= min(window_rtts) < A
                                )
                            })
                            amb_start = amb_end = None
                            ambiguous_slots.clear()

                        # detection transitions (skip ambiguous)
                        if state == 'normal':
                            if (prev_window_min is not None
                                    and prev_window_min < (A - S)
                                    and curr_min >= A):
                                suspect_time = close_time
                                state = 'suspected'
                        elif state == 'suspected':
                            if curr_min >= A:
                                detect_time = close_time
                                state = 'detected'
                                recovery_slots = set()
                                samples_in_recovery = 0
                            else:
                                state = 'normal'

                        prev_window_min = curr_min

                # clear window
                window_start = None
                window_rtts = []
                window_tss = []
                last_window_ts = None

        # Handle detected-phase recovery
        if state == 'detected':
            recovery_slots.add(_floor(ts * 4))
            if rtt < (A - S):
                recover_time = ts
                recovery_slots.add(_floor(recover_time * 4))
                detection_delay = (
                    detect_time - attack_start_ts if attack_start_ts is not None else None
                )
                attacks.append({
                    'attack_start': attack_start_ts,
                    'suspected': suspect_time,
                    'detected': detect_time,
                    'recovered': recover_time,
                    'duration': len(recovery_slots) * 0.25,
                    'detection_delay': detection_delay,
                    'samples_to_detect': samples_to_detect,
                    'samples_in_recovery': samples_in_recovery
                })
                # reset
                state = 'normal'
                prev_window_min = None
                attack_start_ts = None
                detect_time = None
                suspect_time = None
                recovery_slots.clear()
                samples_to_detect = 0
                samples_in_recovery = 0
            continue

        # Add sample to window
        if window_start is None:
            window_start = _floor(ts * 4) / 4.0
        window_rtts.append(rtt)
        window_tss.append(ts)
        last_window_ts = ts

    # After loop: finalize detected attack
    if state == 'detected':
        recovery_slots.add(_floor(timestamps[-1] * 4))
        recover_time = timestamps[-1]
        detection_delay = (
            detect_time - attack_start_ts if attack_start_ts is not None else None
        )
        attacks.append({
            'attack_start': attack_start_ts,
            'suspected': suspect_time,
            'detected': detect_time,
            'recovered': recover_time,
            'duration': len(recovery_slots) * 0.25,
            'detection_delay': detection_delay,
            'samples_to_detect': samples_to_detect,
            'samples_in_recovery': samples_in_recovery
        })

    # Finalize ambiguous for open or unclosed window
    if window_start is not None:
        curr_min = min(window_rtts)
        num_samples = len(window_rtts)
        elapsed = timestamps[-1] - window_start
        qualifies = (
            (num_samples >= N and elapsed <= 60.0)
            or
            (num_samples < N and elapsed >= 60.0 and curr_min < (A - S))
        )
        if qualifies and (A - S) <= curr_min < A:
            slots = {_floor(t*4) for t in window_tss}
            ambiguous_periods.append({
                'start': window_start,
                'end': _ceil(last_window_ts * 4) / 4.0,
                'duration': len(slots) * 0.25,
                'sample_count': num_samples
            })

    return { 'attacks': attacks,
             'ambiguous_periods': ambiguous_periods }

In [ ]:
# Test case
test_ts = [(i + random.randint(1, 9)/10)/10 for i in range(1, 201, 1)]
test_rtts = [random.randint(20, 40) for i in range(50)] + [random.randint(120, 140) for i in range(25)] + [random.randint(20, 40) for i in range(50)] \
            + [random.randint(20, 40) for i in range(25)] + [random.randint(120, 140) for i in range(25)] + [random.randint(20, 40) for i in range(25)]
print(detect_attacks_using_variable_windows(test_ts, test_rtts, 100, 20, 10, 5))

In [ ]:
# Test case from real data
test_prefix = "107.139.33.0"
test_ts, test_rtts = detection_phase_samples_varwin[test_prefix]
# test_ts = [t-test_ts[0] for t in test_ts]
# print(df_prefix_optimal_attacks[df_prefix_optimal_attacks["Prefix"] == test_prefix].head(n=5))
print(detect_attacks_using_variable_windows(test_ts, test_rtts, 100, 7, 5, 5))
# pu.scatterplot(test_ts, test_rtts, {"xlim": (0, 3), "ylim": (0, 200)})
for t, r in zip(test_ts, test_rtts):
    if t < 4:
        print(t, r)

In [ ]:
def _perform_detection_per_prefix(args):
    ptype, surge_threshold, prefix = args
    timestamps_s, rtts_ms = detection_phase_samples_varwin[prefix]
    geodesic_threshold = geodesic_thresholds_varwin[prefix]
    out = {}
    
    for attacker in covered_prefixes_varwin[ptype]["passing"][surge_threshold][prefix]:
        absolute_threshold = absolute_thresholds[prefix][attacker]

        out[attacker] = detect_attacks_using_variable_windows(
                timestamps_s, rtts_ms,
                absolute_threshold, surge_threshold, geodesic_threshold,
                MIN_WINDOW_SAMPLE_COUNT
            )
    return ptype, surge_threshold, prefix, out

In [ ]:
# 1) build the list of “tasks” (one task per prefix)
tasks = [
    (ptype, surge_threshold, prefix)
    for ptype in covered_prefixes_varwin
    for surge_threshold in covered_prefixes_varwin[ptype]["passing"]
    for prefix in covered_prefixes_varwin[ptype]["passing"][surge_threshold]
]

In [ ]:
# 2) pre-allocate your result dicts
detection_results_varwin = {
    ptype: {thr: {} for thr in covered_prefixes_varwin[ptype]["passing"]}
    for ptype in covered_prefixes_varwin
}

In [ ]:
total = len(tasks)
completed = 0
report_interval = 10000

# 3) spin up the pool
with mp.Pool(processes=100) as pool:
    for ptype, surge_threshold, prefix, prefix_results in pool.imap_unordered(_perform_detection_per_prefix, tasks):
        # write them back into your nested dict
        detection_results_varwin[ptype][surge_threshold][prefix] = prefix_results
        completed += 1
        if completed % report_interval == 0 or completed == total:
            pct = completed * 100.0 / total
            print(f"Processed {completed}/{total} tasks ({pct:.1f}%)")

In [ ]:
def compute_prefix_fprs(results, ptype):
    fprs = {}
    prefix_fp = {}
    prefix_attacks = {}
    total_fp = {}
    total_attacks = {}
    detection_durations = {}
    recovery_durations = {}
    ambiguous_durations = {}

    for surge in results[ptype]:
        fprs[surge] = []
        prefix_fp[surge] = []
        prefix_attacks[surge] = []
        total_fp[surge] = 0
        total_attacks[surge] = 0
        detection_durations[surge] = []
        recovery_durations[surge]  = []
        ambiguous_durations[surge] = []

        for prefix in results[ptype][surge]:
            prefix_fp_count      = 0
            prefix_attack_count  = 0
            prefix_ambiguous_count = 0
            
            for attacker in results[ptype][surge][prefix]:
                total_attacks[surge] += 1
                prefix_attack_count  += 1
                
                if len(results[ptype][surge][prefix][attacker]["attacks"]) > 0:
                    total_fp[surge] += 1
                    prefix_fp_count += 1
                    for attack in results[ptype][surge][prefix][attacker]["attacks"]:
                        detection_durations[surge].append(attack["detection_delay"])
                        recovery_durations[surge].append(attack["duration"])
                        
                if len(results[ptype][surge][prefix][attacker]["ambiguous_periods"]) > 0:
                    prefix_ambiguous_count += 1
                    for ambiguous in results[ptype][surge][prefix][attacker]["ambiguous_periods"]:
                        ambiguous_durations[surge].append(ambiguous["duration"])

            if prefix_attack_count > 0:
                fprs[surge].append(prefix_fp_count * 100.0 / prefix_attack_count)
                prefix_fp[surge].append(prefix_fp_count)
                prefix_attacks[surge].append(prefix_attack_count)
                    
        fprs[surge] = sorted(fprs[surge])
        detection_durations[surge] = sorted(detection_durations[surge])
        recovery_durations[surge]  = sorted(recovery_durations[surge])
        ambiguous_durations[surge] = sorted(ambiguous_durations[surge])
            
    return fprs, prefix_fp, prefix_attacks, total_fp, total_attacks, detection_durations, recovery_durations, ambiguous_durations

In [ ]:
fprs_ciso_varwin, prefix_fp_ciso_varwin, prefix_attacks_ciso_varwin, total_fp_ciso_varwin, total_attacks_ciso_varwin, \
prefix_detectdur_ciso_varwin, prefix_recoverdur_ciso_varwin, prefix_ambiguousdur_ciso_varwin = compute_prefix_fprs(
    detection_results_varwin, "ciso")
fprs_sico_varwin, prefix_fp_sico_varwin, prefix_attacks_sico_varwin, total_fp_sico_varwin, total_attacks_sico_varwin, \
prefix_detectdur_sico_varwin, prefix_recoverdur_sico_varwin, prefix_ambiguousdur_sico_varwin = compute_prefix_fprs(
    detection_results_varwin, "sico")

print("CISO prefixes:")
for surge in total_fp_ciso_varwin:
    print(f"\tSurge threshold: {surge} ms")
    print(f"\t\tFPR: {mu.rnd(total_fp_ciso_varwin[surge] * 100.0 / total_attacks_ciso_varwin[surge], 4)}%")
    print(f"\t\tMedian recovery time: {np.median(prefix_recoverdur_ciso_varwin[surge])} ms")
    
print("SICO prefixes:")
for surge in total_fp_sico_varwin:
    print(f"\tSurge threshold: {surge} ms")
    print(f"\t\tFPR = {mu.rnd(total_fp_sico_varwin[surge] * 100.0 / total_attacks_sico_varwin[surge], 4)}%")
    print(f"\t\tMedian recovery time: {np.median(prefix_recoverdur_sico_varwin[surge])} s")

In [ ]:
pct_prefixes_varwin = {"ciso": {}, "sico": {}}
pct_fpr_varwin  = {"ciso": {}, "sico": {}}

for sth in range(5, 26, 5):
    pct_prefixes_varwin["ciso"][sth] = []
    pct_prefixes_varwin["sico"][sth] = []
    pct_fpr_varwin["ciso"][sth] = []
    pct_fpr_varwin["sico"][sth] = []

for ptype in pct_prefixes_varwin:
    for surge in pct_prefixes_varwin[ptype]:
        if ptype == "ciso":
            fprs_ptype = fprs_ciso_varwin[surge].copy()
        else:
            fprs_ptype = fprs_sico_varwin[surge].copy()
        for p in range(1, len(fprs_ptype)+1):
            f = fprs_ptype[p-1]
            pct_prefixes_varwin[ptype][surge].append(p * 100.0 / len(fprs_ptype))
            pct_fpr_varwin[ptype][surge].append(f)

In [ ]:
sns_colors = list(sns.color_palette("bright"))
pu.lineplots(
    [pct_prefixes_varwin["ciso"][5], pct_prefixes_varwin["sico"][5]],
    [pct_fpr_varwin["ciso"][5], pct_fpr_varwin["sico"][5]],
    {
        "figsize": (8, 6),
        "colors": [sns_colors[0], sns_colors[1]],
        "linestyles": ["-"],
        "loc": "upper left",
        "curvelabels": ["Clients (5 ms surge)", "Servers (5 ms surge)"],
        "xlim": (-5, 105),
        "ylim": (-5, 105),
        "xlabel": "US Prefixes (%)",
        "ylabel": "False Positives (%)",
        "framealpha": 0.5,
        "plot_path": "plots/prefixes_vs_fprs_surge_varwin.pdf"
})

In [ ]:
fpr_data = []
surges = []
labels = []

for surge in range(5, 26, 5):
    surges.append(surge)
    fpr_data.append(mu.rnd(total_fp_ciso_varwin[surge] * 100.0 / total_attacks_ciso_varwin[surge], 4))
    labels.append(f"{surge}")

pu.barplot(list(range(1, 6, 1)), fpr_data, {
    "figsize": (8, 6),
    "xlabel": "Surge Threshold (ms)", "ylabel": "FP Rate (%)",
    "xticks": labels,
    "colors": [list(sns.color_palette("bright"))[4]],
    "plot_path": "plots/fpr_clients.pdf"
})

In [ ]:
ys = [None] * 4
ys[0] = prefix_recoverdur_ciso_varwin[5]
ys[1] = prefix_recoverdur_sico_varwin[5]
ys[2] = prefix_recoverdur_ciso_varwin[20]
ys[3] = prefix_recoverdur_sico_varwin[20]

pos = []
xtick_pos = []
for i in range(2):
    pos.append(i - 0.2)
    pos.append(i + 0.2)
    xtick_pos.append(i)

fig, ax = plt.subplots(figsize=(8, 6))
boxes = ax.boxplot(ys, positions=pos, widths=0.3, patch_artist=True)
sns_colors = list(sns.color_palette("pastel"))

props = {"legend_1": "Clients", "legend_2": "Servers"}
colors = {props["legend_1"]: sns_colors[0], props["legend_2"]: sns_colors[1]}
methods = list(colors.keys())

for idx, box in enumerate(boxes['boxes']):
    method = methods[idx%len(methods)]
    box.set_facecolor(colors[method])

ax.set_xticks([0, 1])
ax.set_xticklabels(["5 ms", "20 ms"], fontsize=25)

legend_handles = [mpatches.Patch(color=colors[method], label=method) for method in methods]
ax.legend(handles=legend_handles, loc='upper left', facecolor="white", framealpha=0.7)
ax.set_ylabel("Time to recover (s)")
ax.set_xlabel("Surge Threshold")

plt.tight_layout()
plt.show()

In [ ]:
ttr_data = []
surges = []
labels = []

for surge in range(5, 26, 5):
    surges.append(surge)
    ttr_data.append(prefix_recoverdur_ciso_varwin[surge])
    labels.append(f"{surge}")

pu.boxplot(ttr_data, {
    "figsize": (8, 6),
    "xlabel": "Surge Threshold (ms)", "ylabel": "Downtime (s)",
    "xticks": labels,
    "facecolor": "limegreen",
    "plot_path": "plots/downtime_clients.pdf"
})

## Fixed windows-based detection

In [ ]:
df_prefixes_profiling_fixedwindows = pd.read_pickle(prefix_rtt_profiling_fixed_windows_path)
df_prefixes_profiling_fixedwindows.head(n=1)